# Randomized Algorithms

Notes and runnable examples on **randomized algorithms**: methods that flip coins. Injecting randomness buys two things that deterministic code struggles with — **simplicity** (no adversarial worst case to engineer around, because the adversary can't see your coin flips) and **speed** (an O(n) verification can replace an O(n³) recomputation). The price is that a guarantee becomes a *probability*. This notebook draws the central distinction — **Las Vegas** (always correct, random runtime) vs **Monte Carlo** (fixed runtime, possibly wrong) — then builds the canonical primitives from scratch and **proves every probabilistic claim empirically**: reservoir sampling is uniform, the Fisher-Yates shuffle is unbiased (and the classic buggy shuffle is not), and Monte Carlo estimation/verification behaves exactly as the math predicts. Every RNG is seeded, so every number below is reproducible.

**Contents**

1. **Las Vegas vs Monte Carlo** — two ways to be random, and how to reason about each
2. **Reservoir sampling** — uniformly sample k items from a stream of unknown length, one pass
3. **Fisher-Yates shuffle** — the correct unbiased shuffle, and the off-by-one that ruins it
4. **Monte Carlo estimation & verification** — estimating π, and Freivalds' one-sided matrix check
5. **Randomized data structures** — where treaps and skip lists live
6. **When to use what**
7. **Cheat-sheet** — Las Vegas vs Monte Carlo, error & runtime trade-offs

## 1. Las Vegas vs Monte Carlo — two ways to be random

A randomized algorithm uses random bits as part of its logic. There are two fundamentally different bargains it can strike, and naming them is half the battle:

| Flavor | Correctness | Runtime | Mental model |
|---|---|---|---|
| **Las Vegas** | **always correct** | random (a *random variable*) | "keep rolling until you win — you *will* win, eventually" |
| **Monte Carlo** | correct **with high probability** | fixed (bounded) | "roll a fixed number of times — usually right, occasionally wrong" |

You've already met both. Randomized **quickselect** (see the **selection** notebook) is **Las Vegas**: its random pivot makes the runtime a random variable (O(n) expected, O(n²) in a vanishingly unlikely worst case), but the answer is *always* the true k-th element. The skip list and treap (see **probabilistic structures** and **trees**) use randomness for *balance* but stay exact — also Las Vegas in spirit. Bloom filters and count-min sketches, by contrast, are **Monte Carlo**: fixed work, but with a bounded chance of a wrong "yes" or an over-count.

The reasoning tools differ too. For **Las Vegas** you analyze the *expected runtime* (and prove the algorithm always halts with the right answer). For **Monte Carlo** you analyze the *error probability* — and crucially, whether the error is **one-sided** (wrong in only one direction, which you can drive to zero by repeating) or two-sided.

The cleanest way to feel the difference is to solve **one** problem both ways. The problem: find a "marked" item hidden at one position in an array of `n`, by probing random positions.

In [1]:
import random
import statistics

def las_vegas_find(arr, target, rng):
    """Las Vegas: probe random positions UNTIL we hit the target.
    Always returns the correct index; the number of probes is a random variable."""
    probes = 0
    while True:
        probes += 1
        i = rng.randrange(len(arr))
        if arr[i] == target:
            return i, probes            # correctness is guaranteed; time is random

def monte_carlo_find(arr, target, budget, rng):
    """Monte Carlo: probe a FIXED number of times, then give up.
    Fixed runtime; may wrongly report 'absent' (one-sided error: never a false 'present')."""
    for _ in range(budget):
        if arr[rng.randrange(len(arr))] == target:
            return True                 # a 'present' answer is ALWAYS correct
    return False                        # a 'absent' answer MIGHT be wrong

n = 100
arr = list(range(n))                    # the target (42) is present exactly once
print(f'n = {n}, one marked item present')

# --- Las Vegas: always correct; measure the runtime DISTRIBUTION ---
rng = random.Random(0)
trials = 30_000
probe_counts = [las_vegas_find(arr, 42, rng)[1] for _ in range(trials)]
print('\nLas Vegas (always correct):')
print(f'  mean probes  : {statistics.mean(probe_counts):.2f}   (theory = n = {n}, a geometric RV)')
print(f'  min / max    : {min(probe_counts)} / {max(probe_counts)}   (runtime varies wildly, but the answer never does)')

# --- Monte Carlo: fixed runtime; measure the ERROR RATE ---
rng = random.Random(1)
budget = 50
found = sum(monte_carlo_find(arr, 42, budget, rng) for _ in range(trials))
success = found / trials
theory = 1 - (1 - 1 / n) ** budget
print(f'\nMonte Carlo (fixed budget = {budget} probes):')
print(f'  success rate : {success:.3f}   theory 1-(1-1/n)^budget = {theory:.3f}')
print(f'  -> {1 - success:.1%} of the time it wrongly says \"absent\" (one-sided error; it never lies the other way)')
assert abs(success - theory) < 0.01, 'Monte Carlo success rate drifted from theory'
print('measured success rate matches theory within 1% -> OK')

n = 100, one marked item present



Las Vegas (always correct):
  mean probes  : 99.23   (theory = n = 100, a geometric RV)
  min / max    : 1 / 968   (runtime varies wildly, but the answer never does)

Monte Carlo (fixed budget = 50 probes):
  success rate : 0.388   theory 1-(1-1/n)^budget = 0.395
  -> 61.2% of the time it wrongly says "absent" (one-sided error; it never lies the other way)
measured success rate matches theory within 1% -> OK


The contrast is the whole lesson. **Same problem, same random probes** — but the Las Vegas version fixes *correctness* and lets *runtime* float (mean ≈ n probes, a geometric random variable with a long tail), while the Monte Carlo version fixes *runtime* (exactly `budget` probes) and lets *correctness* float (a bounded chance of a wrong "absent").

The Monte Carlo error here is **one-sided** — a "present" answer is always true, only "absent" can be wrong. That's the gift that keeps giving: to shrink one-sided error you just **repeat and combine**. Run the check `t` times and the miss probability drops geometrically. We'll exploit exactly this in **Freivalds' algorithm** in section 4. For now, the takeaway is the vocabulary: *Las Vegas → reason about expected runtime; Monte Carlo → reason about error probability (and whether it's one-sided).*

## 2. Reservoir sampling — a uniform sample from a stream of unknown length

You're reading a stream — log lines, a huge file, a network feed — and you want a **uniform random sample of k items**, but you **don't know n in advance** (it might not fit in memory, or the stream may never end). You can't do `random.sample(list(stream), k)` because materializing the stream is exactly what you're trying to avoid.

**Reservoir sampling** (Algorithm R, Vitter 1985) solves this in **one pass, O(k) memory, no knowledge of n**:

- Keep the first `k` items in a "reservoir".
- For each later item at 0-indexed position `i` (so `i ≥ k`), pick a random `j` in `[0, i]`. If `j < k`, **replace** reservoir slot `j` with the new item; otherwise discard it.

The magic is that when you finish, **every one of the n items is in the reservoir with probability exactly k/n** — perfectly uniform, even though early items had many chances to be evicted and late items had few chances to enter. Those two effects cancel *exactly*. The inductive proof: assume after seeing `i` items each is present with prob `k/i`; show the `(i+1)`-th item enters with prob `k/(i+1)`, and an incumbent survives with prob `(k/i)·(1 − 1/(i+1)) = k/(i+1)`. They match — so uniformity is preserved at every step.

Here it is from scratch — note it touches each item once and never looks back:

In [2]:
import random

def reservoir_sample(stream, k, rng):
    """Uniformly sample k items from an iterable of unknown length, in one pass (Algorithm R)."""
    reservoir = []
    for i, item in enumerate(stream):
        if i < k:
            reservoir.append(item)          # fill the reservoir with the first k
        else:
            j = rng.randrange(i + 1)         # j uniform in [0, i]
            if j < k:
                reservoir[j] = item          # evict slot j, admit the newcomer
    return reservoir

rng = random.Random(0)
# A stream whose length we pretend not to know (a generator, single-pass):
stream = (f'line-{i}' for i in range(1_000_000))
sample = reservoir_sample(stream, 5, rng)
print('5 items sampled from a 1,000,000-line stream in ONE pass, O(k) memory:')
print(' ', sample)

5 items sampled from a 1,000,000-line stream in ONE pass, O(k) memory:
  ['line-250559', 'line-328243', 'line-62377', 'line-140560', 'line-846015']


### The fact-check: every element is selected with frequency ≈ k/n

The claim is uniformity, so we *measure* it. Over many seeded trials we sample `k` from a stream of `n`, tally how often each of the `n` elements ends up selected, and check every element's selection frequency lands at `k/n` within tolerance. If the algorithm had *any* positional bias (favoring early or late items), it would show up here as a sloped histogram.

In [3]:
random.seed(0)
rng = random.Random(0)

n, k, trials = 20, 5, 50_000
expected = k / n                                  # every element should appear this often
selected = [0] * n
for _ in range(trials):
    for item in reservoir_sample(range(n), k, rng):
        selected[item] += 1

freqs = [c / trials for c in selected]
max_dev = max(abs(f - expected) for f in freqs)

print(f'n={n}, k={k}, trials={trials:,}   expected freq per element = k/n = {expected}')
print(f'measured min freq : {min(freqs):.4f}')
print(f'measured max freq : {max(freqs):.4f}')
print(f'max |freq - k/n|  : {max_dev:.4f}')
# Also confirm there is no early-vs-late positional bias.
first_half = sum(freqs[:n // 2]) / (n // 2)
second_half = sum(freqs[n // 2:]) / (n // 2)
print(f'avg freq first half {first_half:.4f}  vs second half {second_half:.4f}  (no positional bias)')
assert max_dev < 0.01, 'a reservoir element drifted from k/n -> not uniform!'
print('\nevery element selected with frequency k/n within 0.01 -> uniform, claim verified')

n=20, k=5, trials=50,000   expected freq per element = k/n = 0.25
measured min freq : 0.2443
measured max freq : 0.2538
max |freq - k/n|  : 0.0057
avg freq first half 0.2505  vs second half 0.2495  (no positional bias)

every element selected with frequency k/n within 0.01 -> uniform, claim verified


The histogram is flat: early items (which had to *survive* many eviction rolls) and late items (which had few chances to *enter*) end up equally likely, exactly as the cancellation in the proof promises. A few practical notes:

- **Weighted variants** exist (Algorithm A-Res / the "exponential jump" of Algorithm L) for when items have different sampling weights, and Algorithm L also skips the per-item coin flip to run in O(k(1 + log(n/k))) time instead of O(n).
- **k = 1** is the everyday special case: "pick one uniformly random line from a file you read once."
- This is the streaming cousin of the **Fisher-Yates shuffle** in the next section — both achieve uniformity by carefully balancing each new item against the incumbents.

## 3. Fisher-Yates shuffle — the correct unbiased shuffle (and the bug that ruins it)

To shuffle a list so that **all n! orderings are equally likely**, the right algorithm is **Fisher-Yates** (the modern in-place form is due to Durstenfeld, 1964). It's what `random.shuffle` uses internally. Walk from the last index down to the first; at position `i`, swap `a[i]` with `a[j]` for a random `j` drawn from `[0, i]` — i.e. **from the unshuffled prefix, including i itself**:

```
for i in range(n-1, 0, -1):
    j = randint(0, i)        # j in [0, i]  <-- the range SHRINKS as i shrinks
    a[i], a[j] = a[j], a[i]
```

Why it's uniform: position `n−1` gets a uniformly random pick of all `n` elements; position `n−2` a uniform pick of the remaining `n−1`; and so on. The number of equally-likely execution paths is `n · (n−1) · … · 1 = n!`, in **bijection** with the permutations. Every ordering, exactly one path.

### The classic bug

The notorious "naive shuffle" looks almost identical but draws `j` from the **full range `[0, n−1]` every time** instead of the shrinking `[0, i]`:

```
for i in range(n):
    j = randint(0, n-1)      # BUG: full range, not [0, i]
    a[i], a[j] = a[j], a[i]
```

This produces `n^n` equally-likely execution paths — and `n^n` is **not divisible by `n!`** for `n > 2`, so the orderings *cannot* come out equally often. Some permutations are systematically favored. The bias is real, subtle, and was shipped in production code (famously a browser-ballot shuffle). Let's catch it red-handed.

In [4]:
import random

def fisher_yates(seq, rng):
    """Correct unbiased in-place shuffle: j drawn from the SHRINKING range [0, i]."""
    a = list(seq)
    for i in range(len(a) - 1, 0, -1):
        j = rng.randint(0, i)               # inclusive [0, i]
        a[i], a[j] = a[j], a[i]
    return tuple(a)

def naive_shuffle(seq, rng):
    """The classic BUGGY shuffle: j drawn from the FULL range [0, n-1] every step."""
    a = list(seq)
    n = len(a)
    for i in range(n):
        j = rng.randint(0, n - 1)           # BUG: full range -> n^n paths, not n!
        a[i], a[j] = a[j], a[i]
    return tuple(a)

# sanity: both return some permutation of the input
rng = random.Random(0)
print('fisher_yates([1,2,3,4]) ->', fisher_yates([1, 2, 3, 4], rng))
print('naive_shuffle([1,2,3,4]) ->', naive_shuffle([1, 2, 3, 4], rng))

fisher_yates([1,2,3,4]) -> (3, 1, 2, 4)
naive_shuffle([1,2,3,4]) -> (3, 4, 1, 2)


### The fact-check: count every permutation over many trials

On `n = 3` there are only `3! = 6` permutations, so we can tally each one exactly. We run both shuffles tens of thousands of times and compare the observed counts against the uniform expectation `trials / 6`. A clean way to quantify "is this uniform?" is the **chi-square statistic** `Σ (observed − expected)² / expected`: near the degrees of freedom (here 5) for a fair process, and blowing up when the distribution is skewed.

In [5]:
from collections import Counter
from itertools import permutations
import math

random.seed(123)
base = [0, 1, 2]
perms = list(permutations(base))
n_perm = math.factorial(len(base))            # 6
trials = 60_000
expected = trials / n_perm

def tally(shuffle_fn, seed):
    rng = random.Random(seed)
    return Counter(shuffle_fn(base, rng) for _ in range(trials))

fy = tally(fisher_yates, 1)
bad = tally(naive_shuffle, 1)

def chi_square(counts):
    return sum((counts[p] - expected) ** 2 / expected for p in perms)

print(f'n=3 -> {n_perm} permutations, {trials:,} trials, expected {expected:.0f} each\n')
print(f'{"permutation":>12} | {"Fisher-Yates":>12} | {"naive (buggy)":>13}')
print('-' * 44)
for p in perms:
    print(f'{str(p):>12} | {fy[p]:>12} | {bad[p]:>13}')

chi_fy, chi_bad = chi_square(fy), chi_square(bad)
fy_dev = max(abs(fy[p] - expected) for p in perms) / expected
bad_dev = max(abs(bad[p] - expected) for p in perms) / expected
print(f'\nFisher-Yates : chi-square = {chi_fy:6.1f}   max deviation from uniform = {fy_dev:5.1%}')
print(f'naive (buggy): chi-square = {chi_bad:6.1f}   max deviation from uniform = {bad_dev:5.1%}')

# Fisher-Yates is uniform; the naive shuffle is provably not.
assert fy_dev < 0.05, 'Fisher-Yates should be ~uniform'
assert chi_fy < 15, 'Fisher-Yates chi-square should be small (~5 dof)'
assert bad_dev > 0.08, 'the naive shuffle should be visibly biased'
assert chi_bad > 100, 'the naive shuffle chi-square should blow up'
print('\nFisher-Yates ~uniform (chi-square ~ 5 dof); naive shuffle visibly biased -> both claims verified')

n=3 -> 6 permutations, 60,000 trials, expected 10000 each

 permutation | Fisher-Yates | naive (buggy)
--------------------------------------------
   (0, 1, 2) |         9879 |          8927
   (0, 2, 1) |         9998 |         11040
   (1, 0, 2) |         9952 |         11235
   (1, 2, 0) |         9984 |         10915
   (2, 0, 1) |        10029 |          8929
   (2, 1, 0) |        10158 |          8954

Fisher-Yates : chi-square =    4.3   max deviation from uniform =  1.6%
naive (buggy): chi-square =  683.7   max deviation from uniform = 12.3%

Fisher-Yates ~uniform (chi-square ~ 5 dof); naive shuffle visibly biased -> both claims verified


The numbers tell the story: Fisher-Yates spreads the 60,000 trials across the six permutations within ~2% of uniform (chi-square near its 5 degrees of freedom), while the one-line-different naive shuffle is off by ~12% with a chi-square in the hundreds — *wildly* non-uniform. The favored permutations are the ones reachable by more of the `n^n` paths; since `n^n` isn't a multiple of `n!`, the imbalance is unavoidable, not a sampling artifact.

Takeaways:

- **Use `random.shuffle`** in real code — it's a correct C-level Fisher-Yates. Hand-roll only to learn, or to shuffle something it can't (e.g. a custom container), and then draw `j` from the *shrinking* range.
- The bug is a perfect illustration of why randomized algorithms demand **empirical verification**: the buggy version *looks* random, passes a casual eyeball test, and still ships a real statistical bias.
- To shuffle a stream of unknown length, you don't shuffle at all — you **reservoir-sample** (previous section).

## 4. Monte Carlo estimation & verification

Two flagship uses of Monte Carlo randomness, both with crisp, checkable predictions:

- **Estimation** — approximate a quantity by random sampling. Error shrinks like **1/√N**: to gain one decimal digit you need ~100× the samples. We'll estimate **π**.
- **Verification** — confirm a claimed result far cheaper than recomputing it, accepting a tiny one-sided error. We'll verify a matrix product `A·B = C` in **O(n²)** instead of the O(n³) it takes to multiply — **Freivalds' algorithm**.

### 4a. Estimating π by sampling

Throw darts uniformly at the unit square `[0,1]²`. The fraction landing inside the quarter-circle `x² + y² ≤ 1` is the ratio of areas, `(π/4)/1 = π/4`. So `π ≈ 4 · (inside / total)`. The estimate is unbiased, and its standard error falls as `1/√N` — the signature convergence rate of Monte Carlo, and also its weakness (slow).

In [6]:
import random
import math

def estimate_pi(n, rng):
    """Monte Carlo estimate of pi: fraction of uniform darts inside the quarter unit circle."""
    inside = 0
    for _ in range(n):
        x, y = rng.random(), rng.random()
        if x * x + y * y <= 1.0:
            inside += 1
    return 4 * inside / n

rng = random.Random(0)
print(f'true pi = {math.pi:.6f}\n')
print(f'{"N":>9} | {"estimate":>9} | {"abs error":>10} | {"~1/sqrt(N)":>11}')
print('-' * 48)
for n in (1_000, 10_000, 100_000, 1_000_000):
    est = estimate_pi(n, rng)
    err = abs(est - math.pi)
    print(f'{n:>9,} | {est:>9.5f} | {err:>10.5f} | {1 / math.sqrt(n):>11.5f}')

# The estimate is consistent: error at N=1e6 should be small (a few x 1/sqrt(N)).
final = estimate_pi(1_000_000, rng)
assert abs(final - math.pi) < 0.01, 'pi estimate too far off at N=1e6'
print(f'\nN=1,000,000 estimate {final:.5f} is within 0.01 of pi -> convergence verified')

true pi = 3.141593

        N |  estimate |  abs error |  ~1/sqrt(N)
------------------------------------------------
    1,000 |   3.12800 |    0.01359 |     0.03162
   10,000 |   3.14320 |    0.00161 |     0.01000
  100,000 |   3.14840 |    0.00681 |     0.00316
1,000,000 |   3.14233 |    0.00074 |     0.00100



N=1,000,000 estimate 3.14075 is within 0.01 of pi -> convergence verified


The error column tracks `1/√N`: each 100× jump in samples buys roughly one extra correct decimal. That `1/√N` rate is *dimension-independent*, which is why Monte Carlo integration wins in high dimensions where grid methods explode — but it's also why π-by-darts is a teaching toy, not a way to actually compute π (you'd need ~10⁸ samples for 4 digits). The lesson is the **convergence law**, not the constant.

### 4b. Verifying a matrix product — Freivalds' algorithm

Now the more striking use. Suppose someone hands you three `n×n` matrices and claims `A·B = C`. Recomputing `A·B` to check costs **O(n³)** (or ~O(n^2.37) with galactic algorithms). **Freivalds' algorithm (1977)** verifies the claim in **O(n²)** with a random vector:

1. Draw a random 0/1 vector `r` of length `n`.
2. Compute `A·(B·r) − C·r`. Each is a matrix-times-*vector*, which is **O(n²)**.
3. If the result is the zero vector, **accept**; otherwise **reject**.

The error is **one-sided**: if `A·B = C`, the check *always* passes. If `A·B ≠ C`, a single round catches it with probability **≥ ½**, so the chance of wrongly accepting after `t` independent rounds is **≤ 2⁻ᵗ** — ten rounds drives it below one in a thousand, all while staying O(t·n²) ≪ O(n³).

In [7]:
import numpy as np

def freivalds(A, B, C, rng, rounds=1):
    """Verify A @ B == C in O(rounds * n^2). One-sided error: never rejects a TRUE product."""
    n = C.shape[1]
    zero = np.zeros(A.shape[0], dtype=C.dtype)
    for _ in range(rounds):
        r = rng.integers(0, 2, size=n)              # random 0/1 vector
        # A @ (B @ r) and C @ r are matrix-VECTOR products: O(n^2) each, no O(n^3) multiply.
        if not np.array_equal(A @ (B @ r) - C @ r, zero):
            return False                            # a mismatch is PROOF that A@B != C
    return True

rng = np.random.default_rng(7)
n = 400
A = rng.integers(0, 10, (n, n))
B = rng.integers(0, 10, (n, n))
C_true = A @ B                                       # the genuine product
C_wrong = C_true.copy()
C_wrong[3, 7] += 1                                   # corrupt a SINGLE entry

print(f'n = {n}')
print('freivalds(A, B, A@B,  rounds=10):', freivalds(A, B, C_true, rng, rounds=10), ' (true product -> always accept)')
print('freivalds(A, B, wrong, rounds=10):', freivalds(A, B, C_wrong, rng, rounds=10), ' (one bad entry -> rejected w.h.p.)')

n = 400
freivalds(A, B, A@B,  rounds=10): True  (true product -> always accept)
freivalds(A, B, wrong, rounds=10): False  (one bad entry -> rejected w.h.p.)


### The fact-check: one-sided error, and miss-rate ≤ 2⁻ᵗ

Two claims to verify empirically. **(1)** A *true* product is *never* rejected — that's a hard guarantee, so we assert it over many seeded rounds. **(2)** For a *wrong* product, the probability of missing it (wrongly accepting) after `t` rounds is `≤ 2⁻ᵗ`. We measure the miss rate over thousands of trials and confirm it tracks the `2⁻ᵗ` bound. (For a single-entry error and a 0/1 vector, a round detects it exactly when `r` is 1 at the bad column — probability exactly ½ — so the measured single-round miss rate should sit right at 0.5.)

In [8]:
rng = np.random.default_rng(123)

# (1) HARD GUARANTEE: a true product is NEVER rejected.
assert all(freivalds(A, B, C_true, rng, rounds=1) for _ in range(2000)), 'rejected a TRUE product!'
print('true product accepted on 2000/2000 single-round checks -> no false rejection (guaranteed)')

# (2) MEASURE the miss rate on a WRONG product vs the 2^-t bound.
print(f'\n{"rounds t":>9} | {"measured miss rate":>18} | {"bound 2^-t":>11}')
print('-' * 44)
trials = 2000
for t in (1, 2, 4, 8):
    misses = sum(freivalds(A, B, C_wrong, rng, rounds=t) for _ in range(trials))  # True == missed
    miss_rate = misses / trials
    bound = 2.0 ** -t
    print(f'{t:>9} | {miss_rate:>18.4f} | {bound:>11.4f}')
    assert miss_rate <= bound + 0.05, f'miss rate at t={t} exceeded the 2^-t bound'

print('\nmiss rate tracks 2^-t (single round ~ 0.5) -> one-sided error, amplifiable -> verified')

true product accepted on 2000/2000 single-round checks -> no false rejection (guaranteed)

 rounds t | measured miss rate |  bound 2^-t
--------------------------------------------


        1 |             0.5050 |      0.5000


        2 |             0.2420 |      0.2500


        4 |             0.0535 |      0.0625


        8 |             0.0040 |      0.0039

miss rate tracks 2^-t (single round ~ 0.5) -> one-sided error, amplifiable -> verified


In [9]:
import timeit

# The payoff: one O(n^2) verification round vs the O(n^3) recompute it replaces.
t_mult = timeit.timeit(lambda: A @ B, number=5) / 5
t_friv = timeit.timeit(lambda: freivalds(A, B, C_true, rng, rounds=1), number=20) / 20
print(f'recompute A @ B (O(n^3))     : {t_mult * 1000:7.2f} ms')
print(f'freivalds 1 round (O(n^2))   : {t_friv * 1000:7.2f} ms')
print(f'-> ~{t_mult / t_friv:.0f}x cheaper per round; even 10 rounds (miss prob < 0.1%) stays far below a recompute')

recompute A @ B (O(n^3))     :   51.92 ms
freivalds 1 round (O(n^2))   :    0.29 ms
-> ~182x cheaper per round; even 10 rounds (miss prob < 0.1%) stays far below a recompute


Exactly as predicted: the true product is *never* rejected, the miss rate on a wrong product halves with each added round (single-round ≈ 0.5, matching the 0/1-vector analysis), and one verification round is **orders of magnitude cheaper** than recomputing the product. This is the Monte Carlo verification pattern in its purest form — **trade a tiny, controllable, one-sided error for an asymptotic speedup**, then amplify the error to nothing by repeating. The same one-sided-error-plus-amplification template underlies the **Miller-Rabin** primality test (see the **number theory** notebook): a "composite" verdict is certain, a "probably prime" verdict is wrong with probability ≤ 4⁻ᵗ.

## 5. Randomized data structures — where they live

Randomness isn't only an algorithmic technique; it's a *design principle* for data structures, where coin flips replace the intricate bookkeeping that keeps a structure balanced. These get full treatment in their own notebooks, cross-referenced here so the map is complete:

| Structure | Idea | Flavor | Covered in |
|---|---|---|---|
| **Skip list** | stack randomized "express lanes" over a sorted list; geometric tower heights keep it O(log n) | Las Vegas (exact, random balance) | **probabilistic structures** |
| **Treap** | a BST keyed by value but heap-ordered by a *random* priority; the randomness makes it balanced in expectation, no rotations-to-rebalance logic | Las Vegas (exact, random balance) | **trees** |
| **Bloom filter** | bit array + k hashes; bounded false-positive rate | Monte Carlo (one-sided error) | **probabilistic structures** |
| **Count-min sketch** | hashed counter grid; over-estimates only | Monte Carlo (one-sided error) | **probabilistic structures** |
| **HyperLogLog** | leading-zero statistics estimate distinct count | Monte Carlo (bounded relative error) | **probabilistic structures** |

The unifying idea: a **skip list** and a **treap** are the *randomized* answers to the same problem an AVL or red-black tree solves with rotations (the **trees** notebook) — they trade a worst-case guarantee for *expected* balance and a dramatically simpler implementation, the same Las Vegas bargain that randomized **quickselect** strikes against median-of-medians in the **selection** notebook. The Monte Carlo structures (Bloom / count-min / HyperLogLog) instead trade exactness for space, with the bounded one-sided error we drew on conceptually in section 1.

## 6. When to use what

| You want... | Reach for | Flavor | Why |
|---|---|---|---|
| A uniform sample of k from a stream of unknown/huge length | **reservoir sampling** | one-pass, O(k) space | no need to know or store n; provably uniform |
| One uniform item from a single-pass stream | reservoir sampling, **k=1** | — | "random line from a file you read once" |
| To shuffle a list uniformly | `random.shuffle` (**Fisher-Yates**) | Las Vegas | C-level, correct, unbiased; don't hand-roll |
| To approximate an integral / quantity | **Monte Carlo estimation** | Monte Carlo | error ~1/√N, dimension-independent; slow but general |
| To verify a costly computation cheaply | **Freivalds**-style check | Monte Carlo, one-sided | O(n²) vs O(n³); amplify error by repeating |
| The k-th element / a single order statistic | randomized **quickselect** | Las Vegas | O(n) expected; always correct (see **selection**) |
| A balanced ordered structure, simply | **skip list** / **treap** | Las Vegas | random balance, no rotation logic (see **trees**, **probabilistic structures**) |
| Membership / counts at scale, small error OK | Bloom / count-min / HLL | Monte Carlo | bounded one-sided error for huge space savings (see **probabilistic structures**) |

**Rules of thumb.**

- **Las Vegas when correctness is non-negotiable** but you can tolerate variable runtime — quickselect, treaps, skip lists. You analyze *expected time*.
- **Monte Carlo when you have a hard time/space budget** and a small, *bounded* error is acceptable — estimation, Freivalds, Bloom filters. You analyze *error probability*, and you love **one-sided** error because repetition drives it to zero.
- **Always seed the RNG** in tests and demos so results are reproducible, and **verify probabilistic claims empirically** — as every section here did. A randomized algorithm that "looks random" can still hide a Fisher-Yates-style bias.

## 7. Cheat-sheet — Las Vegas vs Monte Carlo, error & runtime

| | **Las Vegas** | **Monte Carlo** |
|---|---|---|
| **Correctness** | always correct | correct with high probability |
| **Runtime** | random variable (analyze *expected*) | fixed / bounded |
| **Error** | none | bounded; ideally **one-sided** (amplifiable) |
| **Reduce error by** | n/a (it's exact) | repeating independent rounds |
| **Reduce variance by** | better pivots / priorities | more samples (error ~1/√N for estimation) |
| **Examples** | quickselect, treap, skip list, reservoir, Fisher-Yates | π-estimation, Freivalds, Miller-Rabin, Bloom filter |

<sub>A Las Vegas algorithm can be turned into a Monte Carlo one by *capping its runtime* (stop early, possibly return a wrong/"don't know" answer); a one-sided Monte Carlo algorithm can be *amplified* toward certainty by repetition. n = input size; N = number of samples; t = repetition rounds.</sub>

| Algorithm | Time | Space | Flavor | Guarantee proved above |
|---|:---:|:---:|---|---|
| **Reservoir sampling** | O(n) one pass | O(k) | Las Vegas | each element selected w.p. k/n (measured within 0.01) |
| **Fisher-Yates shuffle** | O(n) | O(1) in-place | Las Vegas | all n! orderings equally likely (chi-square ~ dof) |
| naive shuffle (the bug) | O(n) | O(1) | — | provably biased (n^n not divisible by n!) |
| **Monte Carlo π** | O(N) | O(1) | Monte Carlo | error ~1/√N (each 100× N → +1 digit) |
| **Freivalds' check** | O(t·n²) | O(n) | Monte Carlo, one-sided | true product never rejected; miss rate ≤ 2⁻ᵗ |

**The one-line mental model:** randomness lets you *choose which guarantee to soften*. Soften **runtime** and keep correctness exact → **Las Vegas**. Soften **correctness** (by a bounded, ideally one-sided amount) and keep runtime fixed → **Monte Carlo**. Either way: **seed it, and measure it.**